# 01 - Margin Distributions Overlay

Overlay margin histograms across runs, colored by varying dial.

In [ ]:
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt
import pandas as pd

from playground.runs_io import load_sweep

In [ ]:
EXPERIMENT = "smoke_sweep"
runs_dir = Path.cwd().parent / "runs" / EXPERIMENT
if not runs_dir.exists():
    print(f"No runs at {runs_dir}.")
    df = pd.DataFrame()
else:
    df = load_sweep(runs_dir)
    print(f"loaded {len(df)} runs")

In [ ]:
if not df.empty:
    probe_prefixes = ("margin_", "grad_", "adversarial_", "calibration_", "thickness_", "hessian_", "landscape_")
    summary_suffixes = ("_final", "_best")
    skip_cols = {"run_dir", "experiment_name", "run_id", "extra", "description"}
    config_cols = [c for c in df.columns if not c.endswith(summary_suffixes) and not c.startswith(probe_prefixes) and c not in skip_cols]
    varying = [c for c in config_cols if df[c].nunique() > 1 and c != "seed"]
    print("varying dial(s):", varying)

    if varying:
        dial = varying[0]
        fig, ax = plt.subplots(figsize=(7, 4))
        for _, row in df.iterrows():
            margin_path = Path(row["run_dir"]) / "probes" / "margin.json"
            if not margin_path.exists():
                continue
            margin = json.loads(margin_path.read_text())
            h = margin["histogram"]
            edges = h["edges"]
            centers = [(edges[i] + edges[i + 1]) / 2 for i in range(len(edges) - 1)]
            ax.plot(centers, h["counts"], label=f"{dial}={row[dial]}, seed={row['seed']}", alpha=0.6)
        ax.set_xlabel("margin")
        ax.set_ylabel("count")
        ax.set_title(f"Margin distribution vs {dial}")
        ax.legend(fontsize=7)
        plt.show()